# Train BLIP Baseline

This notebook fine-tunes a BLIP VQA baseline on the processed multi-task JSONL data.

Start with a very small subset first. Full BLIP fine-tuning on CPU can be slow, and checkpoints can use significant disk space.

## 1. Setup

Run this notebook from the repository root, or let the cell below add the project root to `sys.path`.

In [ ]:
from pathlib import Path
import json
import random
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))
PROJECT_ROOT

In [ ]:
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from transformers import BlipForQuestionAnswering, BlipProcessor

torch.__version__

## 2. Config

Keep these values small for the first run. Increase `MAX_SAMPLES` only after the smoke test works.

In [ ]:
METADATA_PATH = PROJECT_ROOT / "data/processed/multitask/validation.jsonl"
MODEL_NAME = "Salesforce/blip-vqa-base"
CHECKPOINT_DIR = PROJECT_ROOT / "outputs/checkpoints/blip_baseline_sample"

MAX_SAMPLES = 100
BATCH_SIZE = 1
EPOCHS = 1
LEARNING_RATE = 5e-6
SEED = 42

DATA_LIMITS = {
    "docvqa": 1667,
    "chartqa": 1667,
    "textvqa": 1666,
}
EXPECTED_TOTAL_SAMPLES = sum(DATA_LIMITS.values())

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
DEVICE

## 3. Prepare Data If Needed

This cell downloads raw samples and builds `data/processed/multitask/validation.jsonl` if the processed file is missing or incomplete. In Colab, this is the cell that recreates the local data instead of requiring data to be committed to GitHub.

In [ ]:
def count_jsonl_lines(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for _ in f)


current_count = count_jsonl_lines(METADATA_PATH)
print(f"Current processed examples: {current_count}")

if current_count < EXPECTED_TOTAL_SAMPLES:
    for dataset_name, limit in DATA_LIMITS.items():
        command = [
            sys.executable,
            str(PROJECT_ROOT / "scripts/prepare_data.py"),
            "--dataset",
            dataset_name,
            "--split",
            "validation",
            "--limit",
            str(limit),
            "--streaming",
        ]
        print("Running:", " ".join(command))
        subprocess.run(command, check=True, cwd=PROJECT_ROOT)

    build_command = [
        sys.executable,
        str(PROJECT_ROOT / "scripts/build_multitask_data.py"),
        "--split",
        "validation",
    ]
    print("Running:", " ".join(build_command))
    subprocess.run(build_command, check=True, cwd=PROJECT_ROOT)
else:
    print("Processed data already exists. Skipping data preparation.")

print(f"Final processed examples: {count_jsonl_lines(METADATA_PATH)}")

In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)

records = []
with METADATA_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

random.shuffle(records)
records = records[:MAX_SAMPLES]

len(records), records[0]

## 4. Dataset and Collate Function

Each sample uses the first ground-truth answer as the training target. This is a simple baseline choice; later experiments can sample among acceptable answers.

In [ ]:
class BlipVQAFinetuneDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        example = self.examples[index]
        answer = example["answers"][0] if example["answers"] else ""
        return {
            "image_path": example["image_path"],
            "question": example["question"],
            "answer": answer,
            "dataset": example["dataset"],
            "task_type": example["task_type"],
        }


processor = BlipProcessor.from_pretrained(MODEL_NAME)


def collate_fn(batch):
    images = [Image.open(PROJECT_ROOT / item["image_path"]).convert("RGB") for item in batch]
    questions = [item["question"] for item in batch]
    answers = [item["answer"] for item in batch]

    inputs = processor(images=images, text=questions, padding=True, return_tensors="pt")
    labels = processor.tokenizer(answers, padding=True, return_tensors="pt").input_ids
    labels[labels == processor.tokenizer.pad_token_id] = -100
    inputs["labels"] = labels

    return inputs


train_dataset = BlipVQAFinetuneDataset(records)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)

len(train_dataset)

## 5. Load Model

In [ ]:
model = BlipForQuestionAnswering.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.train()

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
sum(p.numel() for p in model.parameters())

## 6. Train

This is intentionally minimal full-model fine-tuning. For a more practical experiment, use LoRA/PEFT after the smoke test is stable.

In [ ]:
loss_history = []

for epoch in range(EPOCHS):
    for step, batch in enumerate(train_loader, start=1):
        batch = {key: value.to(DEVICE) for key, value in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_value = float(loss.detach().cpu())
        loss_history.append(loss_value)

        if step % 10 == 0 or step == 1:
            print(f"epoch={epoch + 1} step={step} loss={loss_value:.4f}")

len(loss_history), loss_history[:5], loss_history[-5:]

## 7. Save Checkpoint

The checkpoint is saved under `outputs/`, which is ignored by git.

In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(CHECKPOINT_DIR)
processor.save_pretrained(CHECKPOINT_DIR)

print(f"Saved checkpoint to {CHECKPOINT_DIR}")

## 8. Quick Inference Check

Run one example through the fine-tuned checkpoint to confirm the model can generate.

In [ ]:
model.eval()

example = records[0]
image = Image.open(PROJECT_ROOT / example["image_path"]).convert("RGB")
inputs = processor(image, example["question"], return_tensors="pt")
inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

with torch.inference_mode():
    output_ids = model.generate(**inputs, max_new_tokens=20)

prediction = processor.decode(output_ids[0], skip_special_tokens=True).strip()

print("question:", example["question"])
print("prediction:", prediction)
print("answers:", example["answers"])